In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
import io


In [2]:
from google.colab import files
uploaded = files.upload()

Saving UNSW-NB15_1_partial_binarised_injected .csv to UNSW-NB15_1_partial_binarised_injected .csv


In [39]:
df = pd.read_csv(io.StringIO(uploaded["UNSW-NB15_1_partial_binarised_injected .csv"].decode('utf-8')))

print(df.head())


   sport dport proto state       dur  sbytes  spkts  label
0   1390    53   udp   CON  0.001055     132      2      0
1  33661  1024   udp   CON  0.036133     528      4      0
2   1464    53   udp   CON  0.001119     146      2      0
3   3593    53   udp   CON  0.001209     132      2      0
4  49664    53   udp   CON  0.001169     146      2      0


<ipython-input-39-421d6c963fba>:1: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(uploaded["UNSW-NB15_1_partial_binarised_injected .csv"].decode('utf-8')))


In [40]:
def safe_hex_to_int(x):
    try:
        return int(x, 16)
    except (ValueError, TypeError):
        return 0  # or np.nan, depending on what you want

df['sport'] = df['sport'].apply(safe_hex_to_int)
df['dport'] = df['dport'].apply(safe_hex_to_int)

In [41]:
le = LabelEncoder()


df['proto'] = le.fit_transform(df['proto'])
df['state'] = le.fit_transform(df['state'])
print(df.head())

    sport  dport  proto  state       dur  sbytes  spkts  label
0    5008     83    120      2  0.001055     132      2      0
1  210529   4132    120      2  0.036133     528      4      0
2    5220     83    120      2  0.001119     146      2      0
3   13715     83    120      2  0.001209     132      2      0
4  300644     83    120      2  0.001169     146      2      0


In [42]:
# Features
X = df.drop(columns=['label'])

# Target variable
y = df['label']


In [43]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [44]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Fit the scaler to the training data and transform both training and test sets
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train[:5])


[[ 0.70615027 -0.02193192 -0.08528117  0.54304025  0.00964771 -0.0758866
  -0.34591586]
 [ 1.39300343  0.30090915 -0.08528117  0.54304025 -0.02974395 -0.03999717
   0.15252441]
 [ 0.94528919  0.12838883 -0.08528117  0.54304025 -0.02991947 -0.04727265
   0.07775837]
 [-0.49157709 -0.02205738 -0.08528117  0.54304025 -0.03032613 -0.08963905
  -0.37083787]
 [-0.49157709 -0.02205738 -0.08528117  0.54304025  0.0113452  -0.0758866
  -0.34591586]]


In [45]:
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

In [46]:
classifiers = {
    "SGDClassifier": SGDClassifier(random_state=42),
    "SVM": LinearSVC(random_state=42),
    "KNN": KNeighborsClassifier(),
    "GaussianNB": GaussianNB(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

In [47]:
results = {}

for name, clf in classifiers.items():
    print(f"\n--- {name} ---")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')  # or 'macro'/'micro'

    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    results[name] = {
        "accuracy": acc,
        "f1_score": f1,
    }


--- SGDClassifier ---


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SGDClassifier was fitted with feature names
  warnings.warn(


Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.71      0.82    135681
           1       0.05      0.46      0.09      4544

    accuracy                           0.70    140225
   macro avg       0.51      0.58      0.45    140225
weighted avg       0.95      0.70      0.79    140225

Confusion Matrix:
[[95701 39980]
 [ 2451  2093]]

--- SVM ---


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearSVC was fitted with feature names
  warnings.warn(


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.32      0.48    135681
           1       0.05      0.98      0.09      4544

    accuracy                           0.34    140225
   macro avg       0.52      0.65      0.28    140225
weighted avg       0.97      0.34      0.47    140225

Confusion Matrix:
[[42886 92795]
 [   72  4472]]

--- KNN ---


...

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98    135681
           1       0.00      0.00      0.00      4544

    accuracy                           0.97    140225
   macro avg       0.48      0.50      0.49    140225
weighted avg       0.94      0.97      0.95    140225

Confusion Matrix:
[[135681      0]
 [  4544      0]]

--- AdaBoost ---


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/u

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98    135681
           1       0.00      0.00      0.00      4544

    accuracy                           0.97    140225
   macro avg       0.48      0.50      0.49    140225
weighted avg       0.94      0.97      0.95    140225

Confusion Matrix:
[[135681      0]
 [  4544      0]]


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [48]:
print("\n--- Model Comparison ---")
for model, metrics in results.items():
    print(f"{model} -> Accuracy: {metrics['accuracy']:.4f}, F1 Score: {metrics['f1_score']:.4f}")


--- Model Comparison ---
SGDClassifier -> Accuracy: 0.6974, F1 Score: 0.7949
SVM -> Accuracy: 0.3377, F1 Score: 0.4674
KNN -> Accuracy: 0.9676, F1 Score: 0.9517
GaussianNB -> Accuracy: 0.9676, F1 Score: 0.9517
DecisionTree -> Accuracy: 0.9676, F1 Score: 0.9517
RandomForest -> Accuracy: 0.9676, F1 Score: 0.9517
AdaBoost -> Accuracy: 0.9676, F1 Score: 0.9517
